<a href="https://colab.research.google.com/github/pepestack/CVertidor/blob/master/notebooks/CVertidor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# CVertidor: demostración del agente

Este notebook solo llama al paquete `cvertidor`; toda la lógica está en `src/cvertidor/`. Muestra el flujo completo: generación, revisión humana, conversación con el agente y decisiones adaptativas.

**Requisitos**
- **Local:** `uv sync`, `docker compose up -d` y un `.env` en la raíz del repositorio (ver el README).
- **Colab:** configure los secretos `GOOGLE_API_KEY` y `DB_URL` (una base PostgreSQL con pgvector accesible desde Internet, como Supabase). `OPENROUTER_API_KEY` es opcional.

Use CV ficticios: los modelos reciben el nombre al extraer el CV, y la base guarda los datos originales.

## 1. Preparación

In [ ]:
import importlib.util
import os
from pathlib import Path

# Colab: instala el paquete si todavía no está
if importlib.util.find_spec("cvertidor") is None:
    !pip install -q uv
    !uv pip install -q --system "git+https://github.com/pepestack/CVertidor.git"

# Local: .env, samples/, output/ y logs/ están en la raíz del repositorio
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import uuid

import httpx
from IPython.display import Markdown, display

from cvertidor.config import get_settings
from cvertidor.graph import PendingDecision, ChatReply, chat_agent, graph_mermaid, resume_agent, run_agent
from cvertidor.ingestion.loaders import load_text

settings = get_settings()
HILO = f"demo-{uuid.uuid4().hex[:6]}"


def muestra(nombre: str) -> str:
    """Texto de un archivo de samples/; en Colab se descarga del repositorio"""
    path = Path("samples") / nombre
    if path.exists():
        return load_text(path)
    url = f"https://raw.githubusercontent.com/pepestack/CVertidor/master/samples/{nombre}"
    return httpx.get(url, timeout=30).raise_for_status().text


def mostrar(resultado) -> None:
    if isinstance(resultado, PendingDecision):
        if resultado.preview:
            display(Markdown(resultado.preview))
            print(f"Validación: {resultado.status} por {resultado.source} {resultado.scores}")
        for correccion in resultado.corrections:
            print("Corrección aplicada:", correccion)
        print(f"\nPENDIENTE: {resultado.question}\nOpciones: {resultado.options}")
    elif isinstance(resultado, ChatReply):
        print("Agente:", resultado.text)
        for path in resultado.output_paths:
            print("Guardado:", path)
    elif resultado.discarded:
        print("CV descartado: no se guardó ningún archivo.")
    else:
        display(Markdown(resultado.markdown))
        print(f"Validación: {resultado.verdict.status} por {resultado.verdict.source} {resultado.verdict.scores}")
        for path in resultado.output_paths:
            print("Guardado:", path)
    for aviso in getattr(resultado, "warnings", []):
        print("Aviso:", aviso)


print("Hilo de esta demostración:", HILO)

## 2. Datos de entrada

Pegue su CV y la oferta como texto, o use `load_text("mi_cv.pdf")` para un archivo `.pdf`, `.docx`, `.txt` o `.md` (solo se conserva el texto). Vacíos, se usan los ejemplos ficticios de `samples/`.

In [ ]:
tu_curriculum = """
"""

oferta_de_trabajo = """
"""

cv = tu_curriculum.strip() or muestra("cv_sebastian_navarro.txt")
oferta = oferta_de_trabajo.strip() or muestra("oferta_data_engineer_banco.txt")

## 3. Generar el CV

El agente se detiene antes de guardar y espera una decisión. La pausa queda en Postgres: se puede retomar en otra celda o incluso en otra sesión.

In [ ]:
resultado = run_agent(cv, oferta, settings, thread_id=HILO, on_step=lambda m: print("->", m))
mostrar(resultado)

## 4. Revisar: corregir, aprobar o descartar

Lo que escribe la persona cuenta como dato confirmado, así que también puede agregar información. Las correcciones se acumulan y el resto del CV no cambia.

In [ ]:
resultado = resume_agent(HILO, "corregir", "Quita Looker y agrega Claude Code en habilidades técnicas", settings,
                         on_step=lambda m: print("->", m))
mostrar(resultado)

In [ ]:
resultado = resume_agent(HILO, "aprobar", settings=settings, on_step=lambda m: print("->", m))
mostrar(resultado)

## 5. Conversar con el agente

El agente decide qué herramientas usar: consulta (`search_candidate_experience`, `get_candidate_profile`, `recall_preferences`) y escritura (`edit_cv`, `save_cv_files`, `remember_preference`, `forget_preference`). Toda preferencia que guarde o borre se confirma antes.

In [ ]:
respuesta = chat_agent(HILO, "¿Qué certificaciones tengo? Búscalo en mi CV original.", settings,
                       on_step=lambda m: print("->", m))
mostrar(respuesta)

## 6. Decisiones adaptativas

Otra oferta cambia el plan. El CV y sus fragmentos se reutilizan de la memoria, sin volver a llamar al modelo que los extrajo. `auto_approve=True` omite la revisión de un CV aprobado.

In [ ]:
ingles = run_agent(cv, muestra("job_data_engineer_en.txt"), settings, thread_id=f"{HILO}-en", auto_approve=True)

if isinstance(ingles, PendingDecision):
    print("La validación pidió una decisión:", ingles.question, ingles.issues)
else:
    plan = ingles.plan
    print("CV reutilizado de la memoria:", ingles.cv_from_memory)
    print("Idioma de salida:", plan.output_language, "| traducción:", plan.translate)
    print("Cobertura de requisitos:", plan.skill_coverage)
    print("Prioridad de secciones:", plan.section_priority)
    print("Requisitos ausentes:", plan.missing_requirements)
    display(Markdown(ingles.markdown))

In [ ]:
ajena = run_agent(cv, muestra("oferta_no_relacionada_chef.txt"), settings, thread_id=f"{HILO}-chef", auto_approve=True)

if isinstance(ajena, PendingDecision):
    print("La validación pidió una decisión:", ajena.question, ajena.issues)
else:
    print("Cobertura de requisitos:", ajena.plan.skill_coverage)
    print("Prioridad de secciones:", ajena.plan.section_priority)
    for evento in ajena.audit_events:
        if evento["node"] == "retrieve":
            print(f"Intento {evento['attempt']}: umbral {evento['threshold']}, "
                  f"{evento['work_history']} fragmentos de experiencia -> {evento['decision']}")
    for aviso in ajena.warnings:
        print("Aviso:", aviso)

## 7. Diagrama del grafo

Copie el texto en [mermaid.live](https://mermaid.live) para dibujarlo. Cada ejecución también deja un evento por nodo en `logs/audit.jsonl`, sin datos personales.

In [ ]:
print(graph_mermaid())